In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("Numeric dataset.csv")

In [56]:
x = df.drop("Target", axis=1)
y = df["Target"]
# I dropped target column in x so it selects every other column

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size= 0.2,
    random_state= 100,
    stratify= y
    # stratify makes sure that my target column is balanced, meaning I don't end up with 
    # imbalanced heart disease to no heart disease ratio
)

print(f"Total: {x.shape}")
print(f"Training: {x_train.shape}")
print(f"Testing: {x_test.shape}")


Total: (303, 22)
Training: (242, 22)
Testing: (61, 22)


the printed numbers above tell me that I have correctly split the data. 242 is 80% of 303, and 61 is 20% of 303, so it has been correctly divided. The 15 is the amount of columns I have. 

In [57]:
continuous = ['Age', 'Chest pain', 'Rest BP', 'Chol', 'Max HR', 'Old Peak'] 
categorical = ['Sex', 'Fbs', 'Ex Ang', 'Thal_fixed', 'Thal_normal', 'Thal_reversable', 'Rest ECG_0', 'Rest ECG_1', 'Rest ECG_2', 'Slope_1', 'Slope_2', 'Slope_3', 'Ca_0', 'Ca_1', 'Ca_2', 'Ca_3'] 

x_train_continuous = x_train[continuous]
x_test_continuous = x_test[continuous]

x_train_categorical = x_train[categorical]
x_test_categorical = x_test[categorical]

In [58]:
import numpy as np
from sklearn.preprocessing import StandardScaler

columns = x_train[['Age','Rest BP','Chest pain','Chol','Max HR','Old Peak']]
x_train_continuous = StandardScaler().fit_transform(columns.values)
print(x_train_continuous[:2])


[[ 0.06780645 -0.01667292  1.02553083  1.98063584 -1.47602908  2.29859074]
 [ 0.40498101 -0.70054536 -0.99500444  0.98428582  0.42962335  0.78496092]]


Code above scales our columns using Z score standardization. I printed out 2 rows just to check that my data is successfully scaled. 

In [59]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


x_train_final = np.hstack([x_train_continuous, x_train_categorical])
x_test_final = np.hstack([x_test_continuous, x_test_categorical])


finaltree = DecisionTreeClassifier(random_state=100,max_depth=4,min_samples_leaf=5)

print("Started training Decision Tree on all columns")
finaltree.fit(x_train_final, y_train) 
print("Training Complete.")

prediciton = finaltree.predict(x_test_final)
accuracy = accuracy_score(y_test, prediciton)

print(f"Decision tree accuracy: {accuracy:.4f}")

Started training Decision Tree on all columns
Training Complete.
Decision tree accuracy: 0.7049


Above is the first model I trained. Accuracy is less than expected but not bad for a first try. max depth and min samples lead helped bring it up as well. 

In [60]:
from sklearn.ensemble import RandomForestClassifier




finalforest = RandomForestClassifier(random_state=42)
print("Training started")
finalforest.fit(x_train_final, y_train)
print("Training Complete")


prediciton = finalforest.predict(x_test_final)
accuracy = accuracy_score(y_test, prediciton)


print(f"Random Forest accuracy: {accuracy:.4f}")


Training started
Training Complete
Random Forest accuracy: 0.7049


Second model is done. Not sure if the two models having the same exact accuracy mean i did something wrong (since i used the same x and y train and test values??).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


x_train_final = np.hstack([x_train_continuous, x_train_categorical])
x_test_final = np.hstack([x_test_continuous, x_test_categorical])


finaltree = DecisionTreeClassifier(random_state=100,max_depth=4,min_samples_leaf=5)

print("Started training Decision Tree on all columns")
finaltree.fit(x_train_final, y_train) 
print("Training Complete.")

prediction = finaltree.predict(x_test_final)
accuracy = accuracy_score(y_test, prediciton)

print(f"Decision tree accuracy: {accuracy:.4f}")




print(classification_report(y_test, prediction))
print(confusion_matrix(y_test, prediction))

probabilities = finaltree.predict_proba(x_test_final)[:, 1]
auc = roc_auc_score(y_test, y_proba_rf)
print(f"\nROC-AUC: {auc:.4f}")

Started training Decision Tree on all columns
Training Complete.
Decision tree accuracy: 0.7049
              precision    recall  f1-score   support

           0       0.89      0.52      0.65        33
           1       0.62      0.93      0.74        28

    accuracy                           0.70        61
   macro avg       0.76      0.72      0.70        61
weighted avg       0.77      0.70      0.69        61

[[17 16]
 [ 2 26]]

ROC-AUC: 0.7733


In general, it is not a bad score. Recall for group 0 (no disease) is 0.52, which is a little low meaning the model was mislabeling healthy patients. However, recall for group 1 is 0.93 which is great.

Precision is also not bad, so are f1 scores. The ROC-AUC basically shows me the ratios between true and false positives/negatives. There are 17 patients with no disease that have been labeled with no disease (true negative), 16 patients with no disease that have been labeled with disease (false positive), 2 patients with disease that have been labeled with no diease (false negative), and 26 patients with no disease that have been labeled with no disease (true negative).

From the ROC-AUC, I can deduce that the model truly worked in reducing false negatives, but struggled a little with false positives. I'd expected this since a lot of the patients WITH heart disease were asymptomatic and I though it would somehow affect the predicition, so maybe it did :)
